# US Traffic Accident Risk EDA

This notebook analyzes the US-only data contract used by the project. Records before 2020 support offline pretraining. Records from 2020 onward simulate realtime replay and later retraining. The notebook supports both raw split files with `Severity` and processed feature files with `true_severity`.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 120)

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parents[1]

PROCESS_DIR = PROJECT_ROOT / "data" / "process"
SPLIT_DIR = PROJECT_ROOT / "data" / "split"

OFFLINE_PATH = PROCESS_DIR / "us_train_offline_before_2020.csv"
REPLAY_PATH = SPLIT_DIR / "us_pipeline_from_2020.csv"
SAMPLE_ROWS = 300_000

print(f"Project root: {PROJECT_ROOT}")
print(f"Offline data: {OFFLINE_PATH}")
print(f"Replay data: {REPLAY_PATH}")

In [ ]:
def load_csv(path: Path, label: str) -> pd.DataFrame:
    """Load a bounded sample from a CSV file and label its pipeline role."""
    if not path.exists():
        print(f"Missing {label}: {path}")
        return pd.DataFrame()
    df = pd.read_csv(path, nrows=SAMPLE_ROWS, low_memory=False)
    df["dataset"] = label
    print(f"Loaded {label}: {len(df):,} rows, {len(df.columns):,} columns")
    return df


offline_df = load_csv(OFFLINE_PATH, "before_2020_pretrain")
replay_df = load_csv(REPLAY_PATH, "from_2020_realtime")
display(offline_df.head(3))
display(replay_df.head(3))

In [ ]:
def standardize_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Convert raw or processed US accident columns into one EDA schema."""
    if df.empty:
        return df.copy()
    result = df.copy()

    if "true_severity" not in result.columns and "Severity" in result.columns:
        result["true_severity"] = pd.to_numeric(result["Severity"], errors="coerce")
    if "event_time" not in result.columns and "Start_Time" in result.columns:
        result["event_time"] = pd.to_datetime(result["Start_Time"], errors="coerce")
    elif "event_time" in result.columns:
        result["event_time"] = pd.to_datetime(result["event_time"], errors="coerce")

    rename_map = {
        "Start_Lat": "lat",
        "Start_Lng": "lon",
        "Temperature(F)": "temperature_f",
        "Humidity(%)": "humidity",
        "Wind_Speed(mph)": "wind_speed_mph",
        "Visibility(mi)": "visibility_mi",
    }
    for raw_name, feature_name in rename_map.items():
        if feature_name not in result.columns and raw_name in result.columns:
            result[feature_name] = pd.to_numeric(result[raw_name], errors="coerce")

    if "event_year" not in result.columns and "event_time" in result.columns:
        result["event_year"] = result["event_time"].dt.year
    if "hour" not in result.columns and "event_time" in result.columns:
        result["hour"] = result["event_time"].dt.hour
    if "day_of_week" not in result.columns and "event_time" in result.columns:
        result["day_of_week"] = (result["event_time"].dt.dayofweek + 2).where(result["event_time"].dt.dayofweek < 6, 1)

    required_columns = ["true_severity", "lat", "lon", "event_year", "hour"]
    missing_columns = [column for column in required_columns if column not in result.columns]
    if missing_columns:
        raise ValueError(f"EDA schema is missing required columns: {missing_columns}")
    return result


combined_df = pd.concat([standardize_columns(offline_df), standardize_columns(replay_df)], ignore_index=True)
print(combined_df.shape)
display(combined_df[["dataset", "event_year", "event_time", "true_severity", "lat", "lon", "hour"]].head())

## Data Quality Summary

In [ ]:
quality = combined_df.groupby("dataset").agg(
    rows=("dataset", "size"),
    min_year=("event_year", "min"),
    max_year=("event_year", "max"),
    invalid_severity=("true_severity", lambda s: int((~s.between(1, 4)).sum())),
    missing_lat=("lat", lambda s: int(s.isna().sum())),
    missing_lon=("lon", lambda s: int(s.isna().sum())),
)
display(quality)
display((combined_df.isna().mean().sort_values(ascending=False) * 100).round(2).to_frame("missing_pct").head(30))

## Severity Imbalance

In [ ]:
severity_counts = combined_df["true_severity"].dropna().astype(int).value_counts().sort_index()
display(severity_counts.to_frame("rows"))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].pie(severity_counts.values, labels=severity_counts.index, autopct="%1.1f%%", startangle=90)
axes[0].set_title("true_severity class share")
sns.barplot(x=severity_counts.index.astype(str), y=severity_counts.values, ax=axes[1], color="#4c78a8")
axes[1].set_title("true_severity class counts")
axes[1].set_xlabel("true_severity")
axes[1].set_ylabel("rows")
plt.tight_layout()
plt.show()

## Distributions and Correlation

In [ ]:
numeric_features = [
    "true_severity", "lat", "lon", "event_year", "hour", "day_of_week",
    "weather_code", "temperature_f", "humidity", "wind_speed_mph",
    "visibility_mi", "road_type_code", "is_junction", "has_traffic_signal",
    "is_crossing", "is_roundabout", "is_stop", "is_station", "is_railway", "is_night",
]
available_features = [column for column in numeric_features if column in combined_df.columns]
numeric_df = combined_df[available_features].apply(pd.to_numeric, errors="coerce")

numeric_df.hist(figsize=(18, 14), bins=40, color="#4c78a8", edgecolor="white")
plt.tight_layout()
plt.show()

plt.figure(figsize=(14, 10))
sns.heatmap(numeric_df.corr(), cmap="vlag", center=0, linewidths=0.3)
plt.title("Feature correlation heatmap")
plt.tight_layout()
plt.show()

display(numeric_df.describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]).T.round(3))

## Operational Patterns

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

yearly = combined_df.groupby(["event_year", "dataset"]).size().reset_index(name="rows")
sns.lineplot(data=yearly, x="event_year", y="rows", hue="dataset", marker="o", ax=axes[0, 0])
axes[0, 0].axvline(2020, color="red", linestyle="--", linewidth=1)
axes[0, 0].set_title("Rows by year and split")

hourly = combined_df.groupby("hour").agg(rows=("hour", "size"), avg_severity=("true_severity", "mean")).reset_index()
sns.barplot(data=hourly, x="hour", y="rows", color="#72b7b2", ax=axes[0, 1])
axes[0, 1].set_title("Accident count by hour")

if "weather_code" in combined_df.columns:
    weather = combined_df.groupby("weather_code").agg(avg_severity=("true_severity", "mean")).reset_index()
    sns.barplot(data=weather, x="weather_code", y="avg_severity", color="#f58518", ax=axes[1, 0])
    axes[1, 0].set_title("Average severity by weather_code")

if "road_type_code" in combined_df.columns:
    road = combined_df.groupby("road_type_code").agg(avg_severity=("true_severity", "mean")).reset_index()
    sns.barplot(data=road, x="road_type_code", y="avg_severity", color="#54a24b", ax=axes[1, 1])
    axes[1, 1].set_title("Average severity by road_type_code")

plt.tight_layout()
plt.show()